# Ichimoku Cloud Trend Entry on SPY
## Strategy Brief
The Ichimoku Cloud is a versatile indicator that defines support and resistance, identifies trend direction, gauges momentum, and provides trading signals. This strategy uses the Ichimoku Cloud to identify trend-following entry points on SPY. When the price is above the cloud, it suggests a bullish trend, and when below, a bearish trend. The strategy enters long positions when the price crosses above the cloud and exits when it crosses below, aiming to capture sustained trends.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 - Trading Context
In this phase, we set up the trading context by defining the parameters for the Ichimoku Cloud strategy. These parameters include the standard periods for the Ichimoku Cloud components.

In [ ]:
TENKAN_PERIOD = 9
KIJUN_PERIOD = 26
SENKOU_SPAN_B_PERIOD = 52
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SYMBOL = 'SPY'

## Phase 2 - Data Exploration
We will download historical price data for SPY using yfinance and calculate the Ichimoku Cloud components. The Ichimoku Cloud consists of five lines: Tenkan-sen, Kijun-sen, Senkou Span A, Senkou Span B, and Chikou Span.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(SYMBOL, start=START_DATE, end=END_DATE)

# Calculate Ichimoku Cloud components
data['Tenkan_sen'] = (data['High'].rolling(window=TENKAN_PERIOD).max() + data['Low'].rolling(window=TENKAN_PERIOD).min()) / 2
data['Kijun_sen'] = (data['High'].rolling(window=KIJUN_PERIOD).max() + data['Low'].rolling(window=KIJUN_PERIOD).min()) / 2
data['Senkou_Span_A'] = ((data['Tenkan_sen'] + data['Kijun_sen']) / 2).shift(KIJUN_PERIOD)
data['Senkou_Span_B'] = ((data['High'].rolling(window=SENKOU_SPAN_B_PERIOD).max() + data['Low'].rolling(window=SENKOU_SPAN_B_PERIOD).min()) / 2).shift(KIJUN_PERIOD)
data['Chikou_span'] = data['Close'].shift(-KIJUN_PERIOD)

# Plot Ichimoku Cloud
data[['Close', 'Tenkan_sen', 'Kijun_sen', 'Senkou_Span_A', 'Senkou_Span_B']].plot(figsize=(14, 7))
plt.fill_between(data.index, data['Senkou_Span_A'], data['Senkou_Span_B'], where=data['Senkou_Span_A'] >= data['Senkou_Span_B'], color='lightgreen', alpha=0.3)
plt.fill_between(data.index, data['Senkou_Span_A'], data['Senkou_Span_B'], where=data['Senkou_Span_A'] < data['Senkou_Span_B'], color='lightcoral', alpha=0.3)
plt.title('SPY with Ichimoku Cloud')
plt.show()

## Phase 3 - Strategy Engineering
We will create a signal series based on the Ichimoku Cloud. A long signal is generated when the closing price crosses above the cloud, and an exit signal when it crosses below.

In [ ]:
# Generate signals
signals = pd.DataFrame(index=data.index)
signals['signal'] = 0
signals['signal'][KIJUN_PERIOD:] = np.where(data['Close'][KIJUN_PERIOD:] > data['Senkou_Span_A'][KIJUN_PERIOD:], 1, 0)
signals['signal'] = np.where(data['Close'] < data['Senkou_Span_B'], 0, signals['signal'])

# Entry/Exit logic
signals['positions'] = signals['signal'].diff()

## Phase 4 - Coding & Backtesting
We will backtest the strategy by shifting the positions, calculating daily returns, and plotting the equity curve.

In [ ]:
# Backtesting
initial_capital = float(100000.0)
positions = pd.DataFrame(index=signals.index).fillna(0.0)
positions['SPY'] = 100 * signals['signal']
portfolio = positions.multiply(data['Adj Close'], axis=0)
pos_diff = positions.diff()
portfolio['holdings'] = (positions.multiply(data['Adj Close'], axis=0)).sum(axis=1)
portfolio['cash'] = initial_capital - (pos_diff.multiply(data['Adj Close'], axis=0)).sum(axis=1).cumsum()
portfolio['total'] = portfolio['cash'] + portfolio['holdings']
portfolio['returns'] = portfolio['total'].pct_change()

# Plot equity curve
portfolio['total'].plot(figsize=(10, 6))
plt.title('Equity Curve')
plt.show()

## Phase 5 - Performance Evaluation
We will evaluate the performance of the strategy using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We will compare these to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(portfolio):
    # Calculate CAGR
    n_years = (portfolio.index[-1] - portfolio.index[0]).days / 365.25
    cagr = (portfolio['total'][-1] / portfolio['total'][0]) ** (1/n_years) - 1
    
    # Calculate Sharpe ratio
    sharpe_ratio = (portfolio['returns'].mean() / portfolio['returns'].std()) * np.sqrt(252)
    
    # Calculate Sortino ratio
    downside_returns = portfolio['returns'][portfolio['returns'] < 0]
    sortino_ratio = (portfolio['returns'].mean() / downside_returns.std()) * np.sqrt(252)
    
    # Calculate Calmar ratio
    max_drawdown = ((portfolio['total'].cummax() - portfolio['total']).max() / portfolio['total'].cummax().max())
    calmar_ratio = cagr / max_drawdown
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown = calculate_performance_metrics(portfolio)

# Buy-and-hold comparison
buy_and_hold_cagr = (data['Adj Close'][-1] / data['Adj Close'][0]) ** (1/n_years) - 1

# Display results
print(f"CAGR: {cagr:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Buy-and-Hold CAGR: {buy_and_hold_cagr:.2%}")

## Phase 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes the Ichimoku Cloud, and prints today's signal for potential trading decisions.

In [ ]:
def get_latest_signal():
    recent_data = yf.download(SYMBOL, start=pd.to_datetime('today') - pd.Timedelta(days=60), end=pd.to_datetime('today'))
    
    # Calculate Ichimoku Cloud components
    recent_data['Tenkan_sen'] = (recent_data['High'].rolling(window=TENKAN_PERIOD).max() + recent_data['Low'].rolling(window=TENKAN_PERIOD).min()) / 2
    recent_data['Kijun_sen'] = (recent_data['High'].rolling(window=KIJUN_PERIOD).max() + recent_data['Low'].rolling(window=KIJUN_PERIOD).min()) / 2
    recent_data['Senkou_Span_A'] = ((recent_data['Tenkan_sen'] + recent_data['Kijun_sen']) / 2).shift(KIJUN_PERIOD)
    recent_data['Senkou_Span_B'] = ((recent_data['High'].rolling(window=SENKOU_SPAN_B_PERIOD).max() + recent_data['Low'].rolling(window=SENKOU_SPAN_B_PERIOD).min()) / 2).shift(KIJUN_PERIOD)
    
    # Determine today's signal
    if recent_data['Close'][-1] > recent_data['Senkou_Span_A'][-1]:
        print('Long Position')
    elif recent_data['Close'][-1] < recent_data['Senkou_Span_B'][-1]:
        print('Short Position')
    else:
        print('No Position')

get_latest_signal()